# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srilaya30/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [45]:
from pathlib import Path

repo_root = Path("/content/flyrank-ml-internship")

if not repo_root.exists():
    !git clone -q https://github.com/Srilaya30/flyrank-ml-internship.git
import pandas as pd
import numpy as np

repo_root = Path("/content/flyrank-ml-internship")
data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"

print("Dataset exists:", data_path.exists())

df = pd.read_csv(data_path)

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Dataset shape:", df.shape)
print("Target distribution:")
print(df["is_declining_label"].value_counts())

Dataset exists: True
Dataset shape: (30000, 45)
Target distribution:
is_declining_label
1    16262
0    13738
Name: count, dtype: int64


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — Content Lifecycle: Growing vs Declining

The FlyRank research paper reports that growing pages were younger than declining pages in its dataset. Growing pages had an average age of 185 days, while declining pages had an average age of 228 days. Word count was almost the same between the two groups, so the paper presents age as the clearer difference rather than content length.

Methodology question: The paper separates pages into growing and declining groups based on traffic direction. I would want to confirm exactly how the trend label is constructed and whether the validation design accounts for repeated pages, clients, and time when interpreting the difference.

Finding 2 — Refreshing Older Pages

The FlyRank research paper reports a held-out test summary in which 7 of 9 refresh strata showed statistically significant refresh lift. It also reports a median effect of +588 impressions for refreshed versus stale pages among pages older than 180 days, with a 95% confidence interval of [548, 634].

Methodology question: Because this is presented as a refresh effect from held-out comparisons, I would want to understand how refreshed and stale pages were selected and how the analysis controlled for differences between those groups. This matters because an observed difference after refresh does not automatically prove that refresh alone caused the improvement.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

I use a client-grouped 80/20 train-test split. Pages from the same client are kept together so that the model is evaluated on clients it did not see during training. This reduces the risk that client-specific patterns make the test result look better than it would be on unseen clients.

In [28]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [29]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=df["client_id"])
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

print("Training rows:", len(train_idx))
print("Testing rows:", len(test_idx))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Client overlap: 0


In [30]:
feature_cols = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[feature_cols].copy()
y = df["is_declining_label"].copy()

print("Number of features:", len(feature_cols))
print("X shape:", X.shape)
print("y shape:", y.shape)

Number of features: 29
X shape: (30000, 29)
y shape: (30000,)


In [31]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=df["client_id"])
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

print("Training rows:", len(train_idx))
print("Testing rows:", len(test_idx))
print("Training clients:", len(train_clients))
print("Testing clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

Training rows: 23837
Testing rows: 6163
Training clients: 25
Testing clients: 7
Client overlap: 0


In [32]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print("Training missing values:", X_train.isna().sum().sum())
print("Testing missing values:", X_test.isna().sum().sum())

Training missing values: 0
Testing missing values: 0


In [33]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Model training completed.")

Model training completed.


In [34]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(scores)[::-1][:k]
    return np.mean(np.asarray(y_true)[order])

model_p20 = precision_at_k(
    y_test.to_numpy(),
    y_prob,
    20
)

print("Random Forest Precision@20:", round(model_p20, 4))

Random Forest Precision@20: 1.0


In [35]:
baseline_test = df.iloc[test_idx].copy()

baseline_test["staleness_score"] = (
    baseline_test["days_since_last_update"]
    .rank(method="average", pct=True)
)

baseline_test["ctr_opportunity_score"] = (
    1 - baseline_test["ctr"].rank(method="average", pct=True)
)

baseline_test["baseline_score"] = (
    0.60 * baseline_test["staleness_score"]
    + 0.40 * baseline_test["ctr_opportunity_score"]
)

baseline_p20 = precision_at_k(
    baseline_test["is_declining_label"].to_numpy(),
    baseline_test["baseline_score"].to_numpy(),
    20
)

print("Baseline Precision@20:", round(baseline_p20, 4))

Baseline Precision@20: 0.45


In [36]:
comparison = pd.DataFrame({
    "Approach": [
        "Week-4 baseline",
        "Random Forest"
    ],
    "Precision@20": [
        baseline_p20,
        model_p20
    ]
})

print(comparison.to_string(index=False))

       Approach  Precision@20
Week-4 baseline          0.45
  Random Forest          1.00


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

The target is created from trend_direction, where pages with a down trend are assigned a value of 1. I checked the final feature list for direct target-related fields. trend_direction and trend_pct were excluded from the model features so that the target information was not directly given to the model.

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [38]:
target_related_columns = [
    "trend_direction",
    "trend_pct"
]

leakage_found = [
    col for col in feature_cols
    if col in target_related_columns
]

print("Target column:", "is_declining_label")
print("Target source:", "trend_direction == down")
print("Potential target-related features:", leakage_found)

if len(leakage_found) == 0:
    print("PASS: No direct target-related columns are used as model features.")
else:
    print("REVIEW REQUIRED: Target-related columns detected.")

Target column: is_declining_label
Target source: trend_direction == down
Potential target-related features: []
PASS: No direct target-related columns are used as model features.


In [39]:
results = df.iloc[test_idx][
    ["content_id", "client_id", "trend_direction"]
].copy()

results["actual"] = y_test.to_numpy()
results["predicted"] = y_pred
results["probability_declining"] = y_prob
results["correct"] = results["actual"] == results["predicted"]

print("Total test rows:", len(results))
print("Errors:", (~results["correct"]).sum())
print("Error rate:", round((~results["correct"]).mean(), 4))

Total test rows: 6163
Errors: 768
Error rate: 0.1246


In [40]:
false_positives = results[
    (results["actual"] == 0) &
    (results["predicted"] == 1)
]

false_negatives = results[
    (results["actual"] == 1) &
    (results["predicted"] == 0)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

False positives: 432
False negatives: 336


In [41]:
errors = results[results["correct"] == False].copy()

print(
    errors.sort_values(
        "probability_declining",
        ascending=False
    ).head(10).to_string(index=False)
)

          content_id         client_id trend_direction  actual  predicted  probability_declining  correct
content_8f1409b2674e client_8527a891e2          stable       0          1                  0.825    False
content_816d77e36e14 client_8527a891e2          stable       0          1                  0.805    False
content_09227e80fef5 client_8527a891e2          stable       0          1                  0.765    False
content_cf11e5bef79f client_e629fa6598          stable       0          1                  0.760    False
content_15e0e6081fe9 client_f369cb89fc          stable       0          1                  0.755    False
content_093761113c66 client_8527a891e2          stable       0          1                  0.745    False
content_4d9f36001f06 client_8527a891e2          stable       0          1                  0.745    False
content_94fa1e16f2b8 client_4e07408562          stable       0          1                  0.735    False
content_50e211c8623a client_f369cb89fc        

In [42]:
feature_importance = pd.Series(
    model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("Top 10 features:")
print(feature_importance.head(10))

Top 10 features:
impressions_prev_30d     0.204260
impressions_last_30d     0.169643
impressions_90d          0.079278
avg_position             0.057535
days_with_impressions    0.050561
content_age_days         0.048926
sessions_last_30d        0.028343
word_count               0.027802
char_count               0.027741
ctr                      0.025489
dtype: float64


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Original claim: The Random Forest predicts declining content better than the Week-4 baseline.

Safer claim: On the client-grouped test split, the Random Forest achieved a measured Precision@20 of 1.00 compared with 0.45 for the Week-4 baseline. This is an observed and directional result from this test set and can be used as decision support for prioritizing content review. It does not prove that the model will perform the same way on future data or that the identified signals cause content to decline.

In [44]:
print("Final ML-09 checks")
print("Features:", len(feature_cols))
print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Client overlap:", len(train_clients & test_clients))
print("Model Precision@20:", round(model_p20, 4))
print("Baseline Precision@20:", round(baseline_p20, 4))
print("Errors:", int((~results["correct"]).sum()))
print("Leakage columns:", leakage_found)

Final ML-09 checks
Features: 29
Train rows: 23837
Test rows: 6163
Client overlap: 0
Model Precision@20: 1.0
Baseline Precision@20: 0.45
Errors: 768
Leakage columns: []


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.